# Outputs for economic analyses
So far, we've considered some of the features of the disease X epidemic to justify the use of epidemiological models in general, but haven't produced any results that involve economics. In this notebook, we'll use a similar model to the one introduced in the previous two to think about how we might assign costs to a infection-related health state or process within our simulation. Costs may be associated with either modelled states or with transition processes within the model.

We'll again start out by loading some packages, which you can again ignore unless you're interested.

In [ ]:
!pip install git+https://github.com/monash-emu/summer3wip.git@ab4f260aa23d01d1ee50347ea8990c54783ec074

## Capturing the state of being in hospital
In the model in this notebook, we're going to incorporate new structure to represent people in hospital. 


We have done this for simplicity, although there are significant issues with constructing the model this way - in particular, that there is no delay between infection and hospitalisation. This is clearly not realistic, and this delay could be included in the model relatively easily, but we'll keep it as simple as possible for now. This is all just setting up a model that we can use to explain the principles we're interested in later in the notebook.

![](../images/sir_hosp.svg)

In [ ]:
import numpy as np
import pandas as pd

pd.options.plotting.backend = "plotly"

from summer3.graph import defer, Parameter, CompartmentValues
from summer3.epi import (
    Stratification,
    CompartmentMap,
    ManagedArray,
    CategoryData,
    TransitionFlow,
    CompartmentalEpiModel,
)

In [ ]:
population = 1e6
seed = 1.0
start_time = 0
end_time = 50
model_comps = ["susceptible", "infectious", "recovered"]
infect_comps = ["infectious"]
disease_state = Stratification("disease_state", model_comps)
humans = CompartmentMap.new(disease_state)
epi_model = CompartmentalEpiModel(humans, range(start_time, end_time))
start_pop = [population - seed, seed, 0.0]
epi_model.set_initial_population(
    CategoryData(disease_state.categories(), np.array(start_pop))
)


def infection_process(
    compartment_values: ManagedArray,
    contact_rate: float,
    infectious_compartments: tuple,
):
    n_infectious = compartment_values.query(infectious_compartments).sum()
    n_population = compartment_values.sum()
    infectious_prevalence = n_infectious / n_population
    return contact_rate * infectious_prevalence


recovery = TransitionFlow(
    "recovery",
    disease_state["infectious"],
    disease_state["recovered"],
    Parameter("recovery_rate", 0.0),
)
epi_model.add_flow(recovery)

### Stratification
There are a few different ways we could build a model that estimates hospitalisations.
In fact, here, because the hospitalised patients have exactly the same modelled behaviour,
we could actually run the model from notebook 02 and adjust the outputs.
However, it may be clearer to add a state for the hospitalised patients,
which is consistent with the diagram above, 
and if we ever wanted those people to have different modelled epidemiological beahviour,
we could do that more easily with this structure.

So we'll apply a "stratification" to the model, 
but only to the infectious compartment.

In [ ]:
hosp_strata = ["hosp", "non_hosp"]
hosp_strat = Stratification("hospitalisation", hosp_strata)
humans.stratify(hosp_strat, (disease_state, ["infectious"]))

### Specifying the hospitalised fraction
So we now have an additional compartment to represent the hospitalised patients.
However, how should we specify the proportion of cases that are infected.
Again to be explicit, the clearest way to do this is to add two 
infection process flows and then specify the proportion of new episodes
that enter into each hospitalisation stratum.

In [ ]:
hosp_prop = Parameter("hosp_fraction", 0.0)
for strat in hosp_strata:
    strat_prop = hosp_prop if strat == "hosp" else 1.0 - hosp_prop
    split_infection_rate = Parameter("effective_contact_rate", 0.0) * strat_prop
    force_of_infection = defer(infection_process)(
        CompartmentValues, split_infection_rate, disease_state["infectious"]
    )
    infection = TransitionFlow(
        f"infection_{strat}",
        disease_state["susceptible"],
        (disease_state["infectious"], hosp_strat[strat]),
        force_of_infection,
    )
    epi_model.add_flow(infection)

OK, we're ready to run this adjusted model.

In [ ]:
parameters = {"effective_contact_rate": 1.2, "recovery_rate": 0.2, "hosp_fraction": 0.1}
results = epi_model.run(parameters)
occupancy = results["compartments"].to_pandas_df()["infectious_hosp"]
admissions = results["flows"]["infection_hosp"].to_pandas_df()

## Outputs
### Prevalent
First let's look at the number of hospitalisations outputs from the model. As mentioned, we may wish to associate this quantity with costs. If we were to do this, we would want to determine a per-unit time cost, such as a cost per bed-day for hospital occupancy.

In [ ]:
occupancy.plot(labels={"index": "time", "value": "number of people"})

### Incident
Modelled incident quantities are also relevant to economic simulations, because there may often be costs associated with transition processes. For example, when people are admitted to hospital, there may be per-admission costs associated with the process of being managed through the emergency department of the hospitals in which the patients are being managed.

In [ ]:
admissions.plot(labels={"index": "time", "value": "daily new people/admissions"})

### Total costs
Now let's consider how we could build up the total costs of a simulation from the states and transitions we previously identified. We might do this with a micro-costing approach of assigning a price to each of the ingredients associated with these quantities.
This is intended to illustrate the way a dynamic model can be used as the basis for an econometric analysis.

In [ ]:
# Cost per hospital bed day
occupancy_cost = 100.0

# Cost per admission
admission_cost = 50.0

# Costs over time
total_costs_over_time = (
    occupancy.sum() * occupancy_cost + admissions.sum() * admission_cost
)

# Total costs over the simulation interval
total_costs_over_period = total_costs_over_time.sum()
print(
    f"The total cost of hospitalisation over the simulation period is: {round(total_costs_over_period / 1e6)} million."
)

Note that the `occupancy_cost` should represent the costs associated with a patient spending one unit of time (i.e. a day) in hospital. If we used a micro-costing approach, we might build up the `occupancy_cost` based on costs that might include the staff costs of providing care, running the building, etc. per unit time. By contrast, the `admission_cost` should represent the costs associated with a new patient making the transition into the hospital. For this, we might need to determine how long the staff spend working the patient through the admission process (if we know the per-unit time staff costs).